In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
from tqdm import tqdm  
import time

from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
from tqdm import tqdm

In [ ]:
class StockDataset_by_year_rolling(Dataset):
    def __init__(self, base_dir, start_year, end_year, task='regression'):
        self.base_dir = base_dir
        self.years = list(range(start_year, end_year + 1))
        self.task = task
        self.scaler = StandardScaler()
        self.data = []
        self.labels = []
        
        # 加载数据并使用进度条
        for year in tqdm(self.years, desc="Loading Data by Year"):
            data_file = os.path.join(base_dir, f"{year}_df_data.feather")
            label_file = os.path.join(base_dir, f"{year}_real_RET.feather")
            
            # 读取数据和标签
            X = pd.read_feather(data_file)
            y = pd.read_feather(label_file)
            
            # 删除 'permno' 和 'DATE' 列，只保留剩余列作为特征
            X = X.drop(columns=['permno', 'DATE'])
            
            # 如果任务是分类，则将y的值转化为0或1
            if self.task == 'classification':
                y = (y > 0).astype(int)
            
            # 转换为numpy数组并存储
            X = X.values
            y = y.values.reshape(-1, 1)

            # 数据标准化
            if year == self.years[0]:
                X = self.scaler.fit_transform(X)  # 对第一个年份的数据进行拟合和转换
            else:
                X = self.scaler.transform(X)      # 仅对后续年份进行转换

            # 将标准化后的数据存储
            self.data.append(X)
            self.labels.append(y)
        
        # 将数据和标签转换为单个数组
        self.data = torch.tensor(np.vstack(self.data), dtype=torch.float32)
        self.labels = torch.tensor(np.vstack(self.labels), dtype=torch.float32)

        # 打印数据集信息
        #print(f"Data loaded from {start_year} to {end_year}.")
        #print(f"Total samples: {len(self.labels)}")
        #print(f"Feature dimension: {self.data.shape[1]}")
        #print(f"Task: {self.task}")
        

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [ ]:
def create_dataloaders(base_dir, start_year_train, start_year_test, end_year_test, task='regression', batch_size=32):
    train_loaders = []
    test_loaders = []

    # 滚动年份创建数据加载器
    for year in range(start_year_test, end_year_test + 1):
        print(f"Constructing dataset for year {year}...")
        
        # 构建训练集，时间范围为 start_year_train 到 year - 1
        train_dataset = StockDataset_by_year_rolling(base_dir, start_year_train, year - 1, task)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        # 构建测试集，时间范围为 year
        test_dataset = StockDataset_by_year_rolling(base_dir, year, year, task)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        train_loaders.append(train_loader)
        test_loaders.append(test_loader)

    return train_loaders, test_loaders

# 模型训练

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm import tqdm
import pandas as pd
import os
import gc
import shutil
import time



# 检查 GPU 可用性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def calculate_r2_oos(real, predict):
    real = np.array(real)
    predict = np.array(predict)
    assert real.shape == predict.shape 
    valid_mask = ~np.isnan(real)
    numerator = np.sum((real[valid_mask] - predict[valid_mask]) ** 2)
    denominator = np.sum(real[valid_mask] ** 2)
    R2_oos = 1 - (numerator / denominator)
    return R2_oos


from tqdm import tqdm

import time

def train_model(model, train_loader, test_loader, input_dim, num_epochs=50, lr=0.001, patience=10, save_path=None, year=None):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    results = {}

    total_start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        
        train_loss /= len(train_loader.dataset)

        model.eval()
        test_loss = 0.0
        y_test_pred_list = []
        y_test_list = []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                test_loss += loss.item() * X_batch.size(0)
                y_test_pred_list.append(outputs.cpu().numpy())
                y_test_list.append(y_batch.cpu().numpy())
        
        test_loss /= len(test_loader.dataset)
        y_test_pred = np.concatenate(y_test_pred_list).ravel()
        y_test = np.concatenate(y_test_list).ravel()
        test_r2 = calculate_r2_oos(y_test, y_test_pred)

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            best_model_state = model.state_dict()
            results['best_loss'] = best_loss
            results['best_r2'] = test_r2
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    total_training_time = time.time() - total_start_time

    # 加载最佳模型
    if best_model_state:
        model.load_state_dict(best_model_state)

    if save_path:
        year_save_path = os.path.join(save_path, f"{year}_{model.__class__.__name__}.pth")
        torch.save(model.state_dict(), year_save_path)
        return year_save_path, results.get('best_r2', None), total_training_time
    else:
        # 如果不保存模型，仍然返回 R² 和训练时间
        return None, results.get('best_r2', None), total_training_time




# 定义 NN1 模型
class NN1(nn.Module):
    def __init__(self, input_dim, hidden_neurons=64):  # 添加 hidden_neurons 参数
        super(NN1, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons)  # 使用 hidden_neurons 作为隐藏层大小
        self.out = nn.Linear(hidden_neurons, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.out(x)

## NN1

In [ ]:
# 创建和训练模型的年度循环
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016
data_dir = 'data/processed/yearly'

# 定义保存的R²结果和模型路径列表
r2_results = {}
model_paths = {}
final_predictions = []
final_true_values = []

for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='year'):
    train_loader, test_loader = create_dataloaders(data_dir, start_year_train, year, year, task='regression', batch_size=32)
    train_loader = train_loader[0] if isinstance(train_loader, list) else train_loader
    test_loader = test_loader[0] if isinstance(test_loader, list) else test_loader
    
    input_dim = train_loader.dataset[0][0].shape[0]

    print(f"\nTraining NN1 model for year {year}...")
    model_save_path = 'outputs/models/NN1'
    model = NN1(input_dim, hidden_neurons=128)

    # 训练模型
    model_path, current_r2, duration = train_model(model, train_loader, test_loader, input_dim, num_epochs=25, lr=0.001, patience=5, save_path=model_save_path, year=year)

    # 如果是第一个年份，直接保存
    if year == start_year_test:
        best_model_path = model_path
        best_r2 = current_r2
    else:
        # 使用上一年的模型进行验证
        prev_model_path = model_paths[year - 1]
        prev_model = NN1(input_dim, hidden_neurons=128).to(device)
        prev_model.load_state_dict(torch.load(prev_model_path))

        _, prev_r2, _ = train_model(prev_model, train_loader, test_loader, input_dim, num_epochs=1, lr=0.001, patience=1, save_path=None, year=None)
        
        # 比较当前模型和上一年模型的R²
        if current_r2 >= prev_r2:
            best_model_path = model_path
            best_r2 = current_r2
        else:
            # 复制上一年的模型作为今年的模型
            best_model_path = os.path.join(model_save_path, f"{year}_{model.__class__.__name__}.pth")
            shutil.copy(prev_model_path, best_model_path)
            tqdm.write(f"Year {year}: Previous model was better. Copied model from year {year-1}.")

    model_paths[year] = best_model_path
    r2_results[year] = best_r2

    # 记录测试结果
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch).cpu().numpy()
            final_predictions.append(outputs.ravel())
            final_true_values.append(y_batch.numpy().ravel())

    gc.collect()

# 计算最终的 R²
final_predictions = np.concatenate(final_predictions)
final_true_values = np.concatenate(final_true_values)
overall_r2 = calculate_r2_oos(final_true_values, final_predictions)

# 保存最终结果
results_save_path = os.path.join(model_save_path, "final_results.txt")
with open(results_save_path, 'w') as f:
    for year, r2 in r2_results.items():
        f.write(f"Year {year}: R² = {r2:.4f}\n")
    f.write(f"\nOverall R²: {overall_r2:.4f}\n")

print(f"Final results and models saved. Overall R²: {overall_r2:.4f}")

## NN2

In [ ]:
import torch.optim as optim

def run_training_for_years(model, start_year_train, start_year_test, end_year_test, data_dir, model_save_base_path, num_epochs=25, lr=0.03, patience=7, task='regression', batch_size=32):
    r2_results = {}
    model_paths = {}
    final_predictions = []
    final_true_values = []

    model_name = model.__class__.__name__
    model_save_path = os.path.join(model_save_base_path, model_name)
    os.makedirs(model_save_path, exist_ok=True)

    for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='year'):
        # 创建数据加载器
        train_loader, test_loader = create_dataloaders(data_dir, start_year_train, year, year, task=task, batch_size=batch_size)
        train_loader = train_loader[0] if isinstance(train_loader, list) else train_loader
        test_loader = test_loader[0] if isinstance(test_loader, list) else test_loader

        model.apply(reset_weights)  # 重置模型权重
        model = model.to(device)

        print(f"\nTraining {model_name} model for year {year}...")

        # 创建优化器和学习率调度器
        optimizer = optim.Adam(model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)  # 每10个epoch将学习率减少一半

        # 训练模型
        model_path, current_r2, duration = train_model(
            model, train_loader, test_loader, input_dim=84,  # 固定输入维度
            num_epochs=num_epochs, lr=lr, patience=patience,
            save_path=model_save_path, year=year
        )

        scheduler.step()  # 每个epoch后更新学习率

        # 如果是第一个年份，直接保存模型
        if year == start_year_test:
            best_model_path = model_path
            best_r2 = current_r2
        else:
            # 使用上一年的模型进行验证
            prev_model_path = model_paths[year - 1]
            prev_model = copy.deepcopy(model).to(device)
            prev_model.load_state_dict(torch.load(prev_model_path))

            _, prev_r2, _ = train_model(
                prev_model, train_loader, test_loader, input_dim=84,
                num_epochs=1, lr=lr, patience=1, save_path=None, year=None
            )

            # 比较 R² 值
            if current_r2 >= prev_r2:
                best_model_path = model_path
                best_r2 = current_r2
            else:
                # 如果上一年的模型更好，复制上一年的模型作为今年的模型
                best_model_path = os.path.join(model_save_path, f"{year}_{model_name}.pth")
                shutil.copy(prev_model_path, best_model_path)
                tqdm.write(f"Year {year}: Previous model was better. Copied model from year {year-1}.")

        model_paths[year] = best_model_path
        r2_results[year] = best_r2

        # 记录测试结果
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                outputs = model(X_batch).cpu().numpy()
                final_predictions.append(outputs.ravel())
                final_true_values.append(y_batch.numpy().ravel())

        gc.collect()

    # 计算最终的 R²
    final_predictions = np.concatenate(final_predictions)
    final_true_values = np.concatenate(final_true_values)
    overall_r2 = calculate_r2_oos(final_true_values, final_predictions)

    # 保存最终结果
    results_save_path = os.path.join(model_save_path, "final_results.txt")
    with open(results_save_path, 'w') as f:
        for year, r2 in r2_results.items():
            f.write(f"Year {year}: R² = {r2:.4f}\n")
        f.write(f"\nOverall R²: {overall_r2:.4f}\n")

    print(f"Final results and models saved. Overall R²: {overall_r2:.4f}")

In [ ]:
import torch
import torch.nn as nn

# 定义 NN2 模型
class NN2(nn.Module):
    def __init__(self, input_dim, hidden_neurons=128, dropout_rate=0.3):
        super(NN2, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons)
        self.bn1 = nn.BatchNorm1d(hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons // 2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons // 2)
        self.out = nn.Linear(hidden_neurons // 2, 1)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        return self.out(x)

# 创建 NN2 模型实例
input_dim = 87  # 数据维度为 84
nn2_model = NN2(input_dim=input_dim, hidden_neurons=128, dropout_rate=0.3)

# 使用我们写好的 pipeline 进行训练
run_training_for_years(
    model=nn2_model,
    start_year_train=1957,
    start_year_test=1987,
    end_year_test=2016,
    data_dir='data/processed/yearly',
    model_save_base_path='outputs/models',
    lr = 0.0001
)

## NN3

In [ ]:
import torch
import torch.nn as nn

# 定义 NN3 模型
class NN3(nn.Module):
    def __init__(self, input_dim, hidden_neurons1=128, hidden_neurons2=64, hidden_neurons3=32, dropout_rate1=0.3, dropout_rate2=0.4, dropout_rate3=0.5):
        super(NN3, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons1)
        self.bn1 = nn.BatchNorm1d(hidden_neurons1)
        self.fc2 = nn.Linear(hidden_neurons1, hidden_neurons2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons2)
        self.fc3 = nn.Linear(hidden_neurons2, hidden_neurons3)
        self.bn3 = nn.BatchNorm1d(hidden_neurons3)
        self.out = nn.Linear(hidden_neurons3, 1)
        self.dropout1 = nn.Dropout(dropout_rate1)
        self.dropout2 = nn.Dropout(dropout_rate2)
        self.dropout3 = nn.Dropout(dropout_rate3)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        return self.out(x)

# 创建 NN3 模型实例
input_dim = 87  # 数据维度为 84
nn3_model = NN3(input_dim=input_dim, hidden_neurons1=128, hidden_neurons2=64, hidden_neurons3=32, dropout_rate1=0.3, dropout_rate2=0.4, dropout_rate3=0.5)

# 使用新的学习率（例如0.0005）进行训练
run_training_for_years(
    model=nn3_model,
    start_year_train=1957,
    start_year_test=1987,
    end_year_test=2016,
    data_dir='data/processed/yearly',
    model_save_base_path='outputs/models',
    lr=0.03
)

## 1D CNN

In [ ]:
import torch
import torch.nn as nn

# 定义 1D CNN 模型
class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes=1, dropout_rate=0.3):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.fc1 = nn.Linear(128 * (input_dim // 8), 64)  # 计算池化后的特征维度
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # 添加通道维度，转换为 [batch_size, 1, input_dim]
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = torch.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = x.view(x.size(0), -1)  # 展平特征图
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.out(x)


# 创建 1D CNN 模型实例
input_dim = 84  # 数据维度为 84
cnn1d_model = CNN1D(input_dim=input_dim, dropout_rate=0.4)

# 使用我们写好的 pipeline 进行训练
run_training_for_years(
    model=cnn1d_model,
    start_year_train=1957,
    start_year_test=1987,
    end_year_test=2016,
    data_dir='data/processed/yearly',
    model_save_base_path='outputs/models',
    lr=0.001  # 根据经验设置较低的学习率
)

## RNN

In [ ]:
import torch
import torch.nn as nn

# 定义标准 RNN 模型
class StandardRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3):
        super(StandardRNNModel, self).__init__()
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_dim, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        # RNN expects input shape: (batch_size, seq_length, input_dim)
        x, _ = self.rnn(x)  # 输出 x 的形状为 (batch_size, seq_length, hidden_dim)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc(x))
        x = self.dropout(x)
        return self.out(x)

# 创建标准 RNN 模型实例
input_dim = 84  # 数据维度为 84
rnn_model = StandardRNNModel(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3)

# 使用我们写好的 pipeline 进行训练
run_training_for_years(
    model=rnn_model,
    start_year_train=1957,
    start_year_test=1987,
    end_year_test=2016,
    data_dir='data/processed/yearly',
    model_save_base_path='outputs/models',
    lr=0.0005  # 设置较低的学习率以确保稳定训练
)

## LSTM

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate)
        self.fc = nn.Linear(hidden_dim, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x, _ = self.lstm(x)  # x 形状为 (batch_size, seq_length, hidden_dim)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc(x))
        x = self.dropout(x)
        return self.out(x)

    
# 创建 RNN 模型实例
input_dim = 84  # 数据维度为 84
lstm_model = LSTMModel(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3)

# 使用我们写好的 pipeline 进行训练
run_training_for_years(
    model=lstm_model,
    start_year_train=1957,
    start_year_test=1987,
    end_year_test=2016,
    data_dir='data/processed/yearly',
    model_save_base_path='outputs/models',
    lr=0.001  # 设置较低的学习率以确保稳定训练
)

## Transformer

In [ ]:
import torch
import torch.nn as nn

# 定义简化版的 Transformer 模型
class SimpleTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=2, dim_feedforward=256, dropout=0.1):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)  # 输入嵌入层，将输入维度映射到 d_model
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        # x 的形状为 (batch_size, seq_length, input_dim)
        x = self.embedding(x)  # 将输入映射到 d_model 维度
        x = x.permute(1, 0, 2)  # 转换为 (seq_length, batch_size, d_model) 供 Transformer 使用
        x = self.transformer_encoder(x)
        x = x.permute(1, 0, 2)  # 转换回 (batch_size, seq_length, d_model)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.out(x)

# 创建 Transformer 模型实例
input_dim = 84  # 数据维度为 84
transformer_model = SimpleTransformer(input_dim=input_dim, d_model=128, nhead=8, num_layers=2, dim_feedforward=256, dropout=0.1)

# 使用我们写好的 pipeline 进行训练
run_training_for_years(
    model=transformer_model,
    start_year_train=1957,
    start_year_test=1987,
    end_year_test=2016,
    data_dir='data/processed/yearly',
    model_save_base_path='outputs/models',
    lr=0.001  # 设置较低的学习率以确保稳定训练
)

## GNN

In [ ]:
import torch
import torch.nn as nn
import torch_geometric.nn as geom_nn
from torch_geometric.data import Data

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

# 定义简化版的 GNN 模型
class SimpleGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super(SimpleGNN, self).__init__()
        
        # 定义图卷积层
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(input_dim, hidden_dim))  # 第一个卷积层，将 input_dim 映射到 hidden_dim
        
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))  # 中间层
        
        self.fc1 = nn.Linear(hidden_dim, 64)  # 全连接层
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)  # 输出层

    def forward(self, x, edge_index, batch):
        # x 的形状为 (num_nodes, input_dim)
        # edge_index 表示边的连接关系，batch 表示节点所属的图
        
        # 依次通过图卷积层
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = self.dropout(x)
        
        # 对节点进行全局池化，得到每个图的图级表示
        x = global_mean_pool(x, batch)  # (batch_size, hidden_dim)
        
        # 全连接层
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        
        # 输出层
        return self.out(x)


gnn_model = SimpleGNN(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout=0.3)
# 创建 GNN 模型实例



# 使用我们写好的 pipeline 进行训练（需要修改数据加载方式以兼容 GNN）
run_training_for_years(
    model=gnn_model,
    start_year_train=1957,
    start_year_test=1987,
    end_year_test=2016,
    data_dir='data/processed/yearly',
    model_save_base_path='outputs/models',
    lr=0.001  # 设置较低的学习率以确保稳定训练
)